# 1. Build a Tree from a Flat `{id, parent_id}` List
**Difficulty:** 🟡 Medium · **Topic:** Trees / Hash Maps · **Pattern:** hash-map index + one pass

> **DevRev context:** databases store hierarchies *flat* — a `tasks` table where each row has an `id` and a `parent_id`. To render a task tree, a comment thread, or a sub-ticket hierarchy you must **reconstruct the tree** from those flat rows. This is the bread-and-butter of any integration that ingests hierarchical data.

## 💡 Concepts

**Core concept(s):** Index every record by `id` in a **hash map**, then make one pass linking each node to its parent's `children` list.

**Why it applies here:** The flat list gives each row its parent's id but not a pointer to the parent object. A hash map turns "find the parent with this id" from an O(n) scan into an O(1) lookup, so the whole tree builds in one linear pass.

**Key intuition:** First put every node in a lookup table; then attach each node under its parent in O(1). Rows with no parent are the roots.

---

### 📚 What is a Hash Map?
A **hash map** (Python `dict`) turns a key into a slot instantly, so lookup / insert / delete are **O(1)** on average. Here it lets us jump straight from an `id` to its node instead of scanning a list.

---

**Prerequisite knowledge:**
- A `dict` from id → node.
- Handling a **forest** (multiple roots), **orphans** (missing parent), and defensive cycle checks.

## 📝 Problem

Given a flat list of records like `{"id": 3, "parent_id": 1, "title": "..."}`, build the tree(s):
return the **root nodes**, each with a nested `children` list. A `parent_id` of `None` marks a root.
There can be **several roots** (a forest), and a `parent_id` pointing at a missing record (an **orphan**).

**Example**
```
records = [
  {"id": 1, "parent_id": None, "title": "Epic: Login"},
  {"id": 2, "parent_id": 1,    "title": "Backend"},
  {"id": 3, "parent_id": 1,    "title": "Frontend"},
  {"id": 4, "parent_id": 2,    "title": "Auth API"},
]
-> one root (1) with children [2, 3]; node 2 has child [4]
```

> Two approaches: a naive scan-for-parent `O(n²)` and a hash-map one-pass `O(n)`.

### Approach 1 — Scan for Each Parent (worst)

**Idea:** For every node, scan the whole list to find its parent record, then attach it.

**Time:** `O(n²)` — an O(n) scan per node. **Space:** `O(n)`.

In [5]:
from typing import List, Optional

def build_tree_naive(records: List[dict]) -> List[dict]:
    # Wrap each record in a node that also has a children list.
    nodes = {r["id"]: {"id": r["id"], "title": r.get("title"), "children": []} for r in records}
    print(f"ndodes are {nodes}")
    roots = []
    for r in records:
        pid = r.get("parent_id")
        if pid is None:
            roots.append(nodes[r["id"]])       # no parent -> it's a root
            continue
        parent = None
        for other in records:                  # <-- the slow part: scan every record to find the parent
            print(f"other is {other}")
            if other["id"] == pid:
                parent = other
                break
        if parent is not None:
            nodes[pid]["children"].append(nodes[r["id"]])   # attach under the found parent
        else:
            roots.append(nodes[r["id"]])       # parent id doesn't exist (orphan) -> treat as a root
    return roots

In [6]:
records = [
    {"id": 1, "parent_id": None, "title": "Epic: Login"},
    {"id": 2, "parent_id": 1,    "title": "Backend"},
    {"id": 3, "parent_id": 1,    "title": "Frontend"},
    {"id": 4, "parent_id": 2,    "title": "Auth API"},
    {"id": 5, "parent_id": 99,   "title": "Orphan task"},   # missing parent -> becomes a root
]


result = build_tree_naive(records)

ndodes are {1: {'id': 1, 'title': 'Epic: Login', 'children': []}, 2: {'id': 2, 'title': 'Backend', 'children': []}, 3: {'id': 3, 'title': 'Frontend', 'children': []}, 4: {'id': 4, 'title': 'Auth API', 'children': []}, 5: {'id': 5, 'title': 'Orphan task', 'children': []}}
other is {'id': 1, 'parent_id': None, 'title': 'Epic: Login'}
other is {'id': 1, 'parent_id': None, 'title': 'Epic: Login'}
other is {'id': 1, 'parent_id': None, 'title': 'Epic: Login'}
other is {'id': 2, 'parent_id': 1, 'title': 'Backend'}
other is {'id': 1, 'parent_id': None, 'title': 'Epic: Login'}
other is {'id': 2, 'parent_id': 1, 'title': 'Backend'}
other is {'id': 3, 'parent_id': 1, 'title': 'Frontend'}
other is {'id': 4, 'parent_id': 2, 'title': 'Auth API'}
other is {'id': 5, 'parent_id': 99, 'title': 'Orphan task'}


### Approach 2 — Hash-Map Index + One Pass (optimal)

**Idea:** Build an `id → node` map first. Then, in a single pass, look each node's parent up in **O(1)** and append the node to that parent's `children`.

**Time:** `O(n)`. **Space:** `O(n)`.

In [ ]:
from typing import List

def build_tree_fast(records: List[dict]) -> List[dict]:
    # 1) Index every node by id so we can find any parent instantly.
    nodes = {r["id"]: {"id": r["id"], "title": r.get("title"), "children": []} for r in records}
    roots = []
    # 2) One pass: link each node under its parent (O(1) lookup), or record it as a root.
    for r in records:
        node = nodes[r["id"]]
        pid = r.get("parent_id")
        if pid is None:
            roots.append(node)                 # a true root
        elif pid in nodes:
            nodes[pid]["children"].append(node)  # O(1) lookup -> attach under the parent
        else:
            roots.append(node)                 # dangling parent_id (orphan) -> surface as a root
    return roots

def has_cycle(records: List[dict]) -> bool:
    """Defensive check: a parent chain must never loop back on itself."""
    parent = {r["id"]: r.get("parent_id") for r in records}
    for start in parent:
        seen, cur = set(), start
        while cur is not None and cur in parent:
            if cur in seen:                    # revisited an id while walking up -> a cycle
                return True
            seen.add(cur); cur = parent[cur]
    return False

In [ ]:
# Correctness check
records = [
    {"id": 1, "parent_id": None, "title": "Epic: Login"},
    {"id": 2, "parent_id": 1,    "title": "Backend"},
    {"id": 3, "parent_id": 1,    "title": "Frontend"},
    {"id": 4, "parent_id": 2,    "title": "Auth API"},
    {"id": 5, "parent_id": 99,   "title": "Orphan task"},   # missing parent -> becomes a root
]

def child_ids(node):
    return sorted(c["id"] for c in node["children"])

for build in (build_tree_naive, build_tree_fast):
    roots = build(records)
    root_ids = sorted(r["id"] for r in roots)
    print(build.__name__, "-> roots:", root_ids)
    assert root_ids == [1, 5]                 # node 1 (real root) + node 5 (orphan)
    root1 = next(r for r in roots if r["id"] == 1)
    assert child_ids(root1) == [2, 3]
    node2 = next(c for c in root1["children"] if c["id"] == 2)
    assert child_ids(node2) == [4]

# cycle detection
assert has_cycle([{"id": 1, "parent_id": 2}, {"id": 2, "parent_id": 1}]) is True
assert has_cycle(records) is False
print("\nAll tests passed")

## ⏱️ Empirically Checking the Complexities

We time each approach on inputs of growing size `n` and read the **doubling ratio**.

| Theoretical | Ratio `n`→`2n` |
|---|---|
| `O(n)` / `O(V+E)` | ≈ **2×** |
| `O(n log n)`      | ≈ **2×** (slightly more) |
| `O(n²)`           | ≈ **4×** |

In [ ]:
import os, sys
_root = os.getcwd()
for _ in range(5):
    if os.path.exists(os.path.join(_root, "bench_utils.py")): break
    _root = os.path.dirname(_root)
if _root not in sys.path: sys.path.insert(0, _root)
from bench_utils import benchmark

def make_worst_case(n):
    # A deep chain: node i's parent is i-1. The naive scan-for-parent is O(n) per node.
    return ([{'id': i, 'parent_id': (i - 1 if i > 0 else None), 'title': f't{i}'} for i in range(n)],)
solutions = {
    "naive scan O(n^2)": build_tree_naive,
    "hash-map  O(n)  ": build_tree_fast,
}
sizes = [500, 1000, 2000, 4000]

benchmark(solutions, make_worst_case, sizes, plot=True)


## 🧩 Patterns Learned

- **Hash-map index kills a repeated scan:** any "find the record with this id" inside a loop should be an O(1) dict lookup, turning O(n²) into O(n).
- **Build the lookup first, then link:** two clean passes (index, then attach) avoid ordering problems (a child may appear before its parent).
- **Signal:** "reconstruct a hierarchy / tree from flat rows", "adjacency from `{id, parent_id}`".
- **DevRev / related:** rendering task or comment trees, org charts, category hierarchies; LeetCode-style: N-ary tree building, Accounts Merge (union-find variant).
- **Common pitfalls:** (1) assuming parents come before children (they may not — index first); (2) crashing on orphans/cycles instead of handling them; (3) O(n²) child-finding by scanning.